# Getting started: explore features and test an idea

Runs in the dev container (`docker-compose.dev.yml`):
- `get_data_db_conn()` → **dev-postgres**, a restored copy of production (`scripts/dev_refresh_db.sh`)
- `/data/features` → production feature files, **read-only**
- `MLFLOW_TRACKING_URI` → **dev-mlflow** (http://localhost:5002), separate from the production registry

Nothing here can change production.

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from nba.plugins.ml.train import load_training_data
from shared.plugins.db_client import get_data_db_conn

df = load_training_data(os.environ["FEATURES_DIR"])
df["game_date"] = pd.to_datetime(df["game_date"])
print(f"{len(df):,} labeled rows, {df['game_date'].min().date()} → {df['game_date'].max().date()}")
df.head()

## Overs hit rate by prop type

Rows are duplicated per bookmaker/line, so count each player/date/prop once.

In [ ]:
props = df.drop_duplicates(["player_id", "game_date", "prop_type"])
props.groupby("prop_type").agg(
    props=("actual_result", "size"),
    over_hit_rate=("actual_result", "mean"),
    mean_implied=("implied_prob_over", "mean"),
)

## How well does the market price Overs?

If the line were perfectly calibrated, points would sit on the diagonal. Any model has to beat this.

In [ ]:
bins = pd.cut(props["implied_prob_over"], np.arange(0.3, 0.71, 0.05))
calib = props.groupby(bins, observed=True).agg(implied=("implied_prob_over", "mean"), hit=("actual_result", "mean"), n=("actual_result", "size"))

fig, ax = plt.subplots(figsize=(5, 5))
ax.plot([0.3, 0.7], [0.3, 0.7], ls="--", c="grey", lw=1)
ax.scatter(calib["implied"], calib["hit"], s=calib["n"] / 5)
ax.set(xlabel="Implied P(Over) from price", ylabel="Actual Over hit rate", title="Market calibration")
calib

## Try a feature

Add a column computed only from information available before the game, then check whether it explains anything the market doesn't. A quick first look: does the hit rate move across the feature's quantiles, relative to what the price implied?

Once a feature looks promising, confirm it with the walk-forward backtest (`nba/plugins/ml/backtest.py`) before it goes near `features.py`.

In [ ]:
props = props.assign(avg_minus_line=props["rolling_avg_10g"] - props["line"])

feature = "avg_minus_line"
q = pd.qcut(props[feature], 5, duplicates="drop")
props.groupby(q, observed=True).agg(
    n=("actual_result", "size"),
    hit_rate=("actual_result", "mean"),
    implied=("implied_prob_over", "mean"),
).assign(hit_minus_implied=lambda t: t["hit_rate"] - t["implied"])

## Query the database directly

In [ ]:
with get_data_db_conn() as conn:
    recent = pd.read_sql(
        "SELECT game_date, player_name, prop_type, line, edge, actual_result FROM recommendations "
        "WHERE sport = 'NBA' AND rank <= 10 ORDER BY game_date DESC, rank LIMIT 20",
        conn,
    )
recent